In [1]:
# ============================================================
# RCC COLUMN DESIGN - G+1 BUILDING
# Based on IS 456:2000
# Educational / preliminary design
# ============================================================

using Printf

# ------------------------------------------------------------
# STEP 1: INPUTS FROM OTHER BUILDING COMPONENTS
# ------------------------------------------------------------

# Factored axial load transferred to column (kN)
Pu = 900.0

# Factored moments transferred to column (kN-m)
Mux = 40.0
Muy = 25.0

# Material properties
fck = 25.0       # Concrete grade M25 (N/mm²)
fy  = 500.0      # Steel grade Fe500 (N/mm²)

# Trial column dimensions
b = 300.0        # Width (mm)
D = 400.0        # Depth (mm)

# Effective length of column
Le = 3.0         # m

# Clear cover
cover = 40.0     # mm


# ------------------------------------------------------------
# STEP 2: COLUMN CROSS-SECTION
# ------------------------------------------------------------

Ag = b * D

println("\n--- STEP 2: COLUMN SECTION ---")
@printf("Column size = %.0f mm × %.0f mm\n", b, D)
@printf("Gross area = %.0f mm²\n", Ag)


# ------------------------------------------------------------
# STEP 3: CHECK SLENDERNESS
# ------------------------------------------------------------

λx = (Le * 1000) / D
λy = (Le * 1000) / b

println("\n--- STEP 3: SLENDERNESS CHECK ---")
@printf("Slenderness ratio about x-axis = %.2f\n", λx)
@printf("Slenderness ratio about y-axis = %.2f\n", λy)

if λx <= 12 && λy <= 12
    println("Column is treated as a SHORT column.")
else
    println("Column is SLENDER. Additional moment effects must be considered.")
end


# ------------------------------------------------------------
# STEP 4: MINIMUM ECCENTRICITY
# IS 456: e_min = L/500 + D/30, subject to minimum 20 mm
# ------------------------------------------------------------

emin_x = max(Le * 1000 / 500 + D / 30, 20.0)
emin_y = max(Le * 1000 / 500 + b / 30, 20.0)

println("\n--- STEP 4: MINIMUM ECCENTRICITY ---")
@printf("Minimum eccentricity about x-axis = %.2f mm\n", emin_x)
@printf("Minimum eccentricity about y-axis = %.2f mm\n", emin_y)


# ------------------------------------------------------------
# STEP 5: MOMENT DUE TO MINIMUM ECCENTRICITY
# ------------------------------------------------------------

Mmin_x = Pu * emin_x / 1000
Mmin_y = Pu * emin_y / 1000

Mux_design = max(Mux, Mmin_x)
Muy_design = max(Muy, Mmin_y)

println("\n--- STEP 5: DESIGN MOMENTS ---")
@printf("Design Mux = %.2f kN-m\n", Mux_design)
@printf("Design Muy = %.2f kN-m\n", Muy_design)


# ------------------------------------------------------------
# STEP 6: AXIAL CAPACITY OF TIED SHORT COLUMN
#
# Pu = 0.4 fck Ac + 0.67 fy Asc
# ------------------------------------------------------------

Ac = Ag

Asc_required = (
    Pu * 1000 - 0.4 * fck * Ac
) / (0.67 * fy)

# If calculated steel is negative, minimum steel governs
Asc_required = max(Asc_required, 0.008 * Ag)

println("\n--- STEP 6: LONGITUDINAL REINFORCEMENT ---")
@printf("Required steel area = %.2f mm²\n", Asc_required)

# Minimum and maximum reinforcement
Asc_min = 0.008 * Ag
Asc_max = 0.06 * Ag

@printf("Minimum steel area = %.2f mm²\n", Asc_min)
@printf("Maximum steel area = %.2f mm²\n", Asc_max)


# ------------------------------------------------------------
# STEP 7: SELECT LONGITUDINAL BARS
# ------------------------------------------------------------

bar_dia = 20.0
area_bar = π * bar_dia^2 / 4

n_bars = ceil(Int, Asc_required / area_bar)

# Rectangular column requires at least 4 longitudinal bars
n_bars = max(n_bars, 4)

Asc_provided = n_bars * area_bar

println("\n--- STEP 7: BAR SELECTION ---")
@printf("Selected bar diameter = %.0f mm\n", bar_dia)
@printf("Number of bars = %d\n", n_bars)
@printf("Steel area provided = %.2f mm²\n", Asc_provided)

steel_percentage = 100 * Asc_provided / Ag

@printf("Percentage of steel = %.2f %%\n", steel_percentage)


# ------------------------------------------------------------
# STEP 8: CHECK REINFORCEMENT LIMITS
# ------------------------------------------------------------

println("\n--- STEP 8: REINFORCEMENT CHECK ---")

if Asc_provided >= Asc_min
    println("✓ Minimum reinforcement satisfied.")
else
    println("✗ Minimum reinforcement NOT satisfied.")
end

if Asc_provided <= Asc_max
    println("✓ Maximum reinforcement satisfied.")
else
    println("✗ Maximum reinforcement exceeded.")
end


# ------------------------------------------------------------
# STEP 9: AXIAL LOAD CAPACITY WITH PROVIDED STEEL
# ------------------------------------------------------------

Pu_capacity = (
    0.4 * fck * (Ag - Asc_provided)
    + 0.67 * fy * Asc_provided
) / 1000

println("\n--- STEP 9: AXIAL CAPACITY CHECK ---")
@printf("Column axial capacity = %.2f kN\n", Pu_capacity)
@printf("Factored axial load = %.2f kN\n", Pu)

if Pu_capacity >= Pu
    println("✓ Axial load capacity is adequate.")
else
    println("✗ Axial capacity is inadequate.")
end


# ------------------------------------------------------------
# STEP 10: DESIGN OF LATERAL TIES
# ------------------------------------------------------------

tie_dia = max(6.0, bar_dia / 4)

# Maximum spacing of ties
tie_spacing = min(
    16 * bar_dia,
    300.0,
    min(b, D)
)

println("\n--- STEP 10: LATERAL TIES ---")
@printf("Tie diameter = %.0f mm\n", tie_dia)
@printf("Maximum tie spacing = %.0f mm\n", tie_spacing)


# ------------------------------------------------------------
# STEP 11: FINAL DESIGN SUMMARY
# ------------------------------------------------------------

println("\n========================================")
println("        FINAL COLUMN DESIGN")
println("========================================")

@printf("Column size        : %.0f × %.0f mm\n", b, D)
@printf("Concrete           : M%.0f\n", fck)
@printf("Steel              : Fe%.0f\n", fy)
@printf("Factored load Pu   : %.2f kN\n", Pu)
@printf("Design Mux         : %.2f kN-m\n", Mux_design)
@printf("Design Muy         : %.2f kN-m\n", Muy_design)
@printf("Longitudinal bars  : %d bars of %.0f mm\n", n_bars, bar_dia)
@printf("Tie reinforcement  : %.0f mm dia @ %.0f mm c/c\n",
        tie_dia, tie_spacing)

println("========================================")


--- STEP 2: COLUMN SECTION ---
Column size = 300 mm × 400 mm
Gross area = 120000 mm²

--- STEP 3: SLENDERNESS CHECK ---
Slenderness ratio about x-axis = 7.50
Slenderness ratio about y-axis = 10.00
Column is treated as a SHORT column.

--- STEP 4: MINIMUM ECCENTRICITY ---
Minimum eccentricity about x-axis = 20.00 mm
Minimum eccentricity about y-axis = 20.00 mm

--- STEP 5: DESIGN MOMENTS ---
Design Mux = 40.00 kN-m
Design Muy = 25.00 kN-m

--- STEP 6: LONGITUDINAL REINFORCEMENT ---
Required steel area = 960.00 mm²
Minimum steel area = 960.00 mm²
Maximum steel area = 7200.00 mm²

--- STEP 7: BAR SELECTION ---
Selected bar diameter = 20 mm
Number of bars = 4
Steel area provided = 1256.64 mm²
Percentage of steel = 1.05 %

--- STEP 8: REINFORCEMENT CHECK ---
✓ Minimum reinforcement satisfied.
✓ Maximum reinforcement satisfied.

--- STEP 9: AXIAL CAPACITY CHECK ---
Column axial capacity = 1608.41 kN
Factored axial load = 900.00 kN
✓ Axial load capacity is adequate.

--- STEP 10: LATERAL TIE